# Stage 0 — Dataset Ingest

Builds the canonical dataset every later stage reads from.

**Inputs**  `raw/labels_extended/*.json` (12 files) · `TAXONOMY.yaml` · `derived/meta/video_manifest.csv`

**Outputs**
| file | contents |
|---|---|
| `derived/meta/strokes.parquet` | 1457 strokes — corrected, classed, fold-assigned, windows precomputed |
| `derived/meta/events.parquet` | bounce / net / empty_event → Stage 6 |
| `derived/meta/rallies.parquet` | rally-ending outcomes → Phase 2 |
| `derived/meta/folds.json` | the 7 grouped folds + window geometry |

Runs in seconds. Touches no video files.


## 1 · Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Config

Window geometry lives here and is written to `folds.json`, so Stages 2–5 all read the same numbers instead of hardcoding their own.

In [2]:
BASE = "/content/drive/MyDrive/tt_coach"

# --- window geometry (single source of truth for Stages 2-5) -----------------
NATIVE_FPS  = 120
PRE_FRAMES  = 60      # 0.50 s before contact — captures the backswing
POST_FRAMES = 36      # 0.30 s after contact  — captures the follow-through
# Asymmetric on purpose: a block's defining property is the ABSENCE of a
# backswing, so pre-contact carries more class information than follow-through.

import json, re, sys
from pathlib import Path
from collections import Counter

import pandas as pd
import yaml

BASE   = Path(BASE)
LABELS = BASE / "raw/labels_extended"
META   = BASE / "derived/meta"
META.mkdir(parents=True, exist_ok=True)

VIDEOS = [f"game_{i}" for i in range(1, 6)] + [f"test_{i}" for i in range(1, 8)]

assert LABELS.exists(), f"Missing {LABELS} — run the reorganiser first."
print(f"Project : {BASE}")
print(f"Labels  : {len(list(LABELS.glob('*.json')))} files")

Project : /content/drive/MyDrive/tt_coach
Labels  : 12 files


## 3 · Source-annotation corrections

Four defects found in the Step 1 audit. Applied here at ingest — the raw files stay pristine and every fix is auditable in one place.

| defect | count | fix |
|---|---|---|
| `xright_backhand_chop` | 1 | stray `x` → `right_backhand_chop` |
| `back_heavyn` | 1 | trailing `n` → `back_heavy` |
| `point` | 1 | Step-1 placeholder never replaced → drop |
| missing `feet` | 1 | leave null, mask in the aux loss |

In [3]:
PRIMARY_FIX = {
    "xright_backhand_chop": "right_backhand_chop",   # stray 'x' typo
}
LEAN_FIX = {
    "back_heavyn": "back_heavy",                     # trailing 'n' typo
}
DROP_PRIMARY = {
    "point",        # Step-1 placeholder the annotators never replaced
}

EXPECTED_STROKES = 1457
EXPECTED_TECH = {"loop": 583, "serve": 290, "push": 279, "block": 187,
                 "flick": 65, "chop": 31, "smash": 12, "lob": 10}

## 4 · Fold definition

Single-video LOVO leaves four folds with a class at **zero** support (`test_2` has no control; `test_3`/`test_5` no defence), which makes macro-F1 on those folds meaningless.

Grouping the thin test videos gives every fold all four classes and ≥152 strokes, while still guaranteeing **no video ever spans train and validation**.

In [4]:
FOLDS = {
    "A": ["game_1"],
    "B": ["game_2"],
    "C": ["game_3"],
    "D": ["game_4"],
    "E": ["game_5"],
    "F": ["test_1", "test_4"],
    "G": ["test_2", "test_3", "test_5", "test_6", "test_7"],
}
VIDEO2FOLD = {v: f for f, vs in FOLDS.items() for v in vs}

for f, vs in FOLDS.items():
    print(f"  fold {f}: {', '.join(vs)}")

  fold A: game_1
  fold B: game_2
  fold C: game_3
  fold D: game_4
  fold E: game_5
  fold F: test_1, test_4
  fold G: test_2, test_3, test_5, test_6, test_7


## 5 · Load taxonomy & video manifest

In [5]:
TAX = yaml.safe_load((BASE / "TAXONOMY.yaml").read_text())
TECH2CLASS = {t: c for c, s in TAX["classes"].items() for t in s["techniques"]}

SIDES = TAX["label_format"]["side_values"]
HIGHS = TAX["label_format"]["high_level_values"]
TECHS = TAX["label_format"]["technique_values"]

STROKE_RE = re.compile(
    rf"^({'|'.join(SIDES)})_({'|'.join(HIGHS)})_({'|'.join(TECHS)})$")
RALLY_RE = re.compile(
    r"^(left|right)_(out|net|winner|not_hitting_ball|double_bounce|"
    r"miss_on_own_side)$")
PLAIN_EVENTS = {"bounce", "net", "empty_event"}

vm_path = META / "video_manifest.csv"
assert vm_path.exists(), "video_manifest.csv missing — run STEP 2 first."
vm = pd.read_csv(vm_path).set_index("video_id")
NFRAMES = vm["nb_frames"].to_dict()

print(f"Taxonomy v{TAX.get('version')}: "
      + ", ".join(f"{c}({len(s['techniques'])})" for c, s in TAX["classes"].items()))
print(f"Manifest: {len(NFRAMES)} videos, {sum(NFRAMES.values()):,} total frames")

Taxonomy v1: serve(1), attack(3), control(1), defence(3)
Manifest: 12 videos, 644,199 total frames


## 6 · Parse all 12 label files

Splits each file into three layers: typed strokes, plain events, and rally outcomes.

In [6]:
strokes, events, rallies = [], [], []
applied = Counter()
unknown = Counter()

for vid in VIDEOS:
    p = LABELS / f"{vid}.json"
    if not p.exists():
        sys.exit(f"Missing label file: {p}")
    data = json.loads(p.read_text())

    for frame_str, raw in data.items():
        parts = str(raw).strip().split()
        if not parts:
            continue
        primary = parts[0]
        frame = int(frame_str)

        if primary in PRIMARY_FIX:
            applied[f"primary: {primary} -> {PRIMARY_FIX[primary]}"] += 1
            primary = PRIMARY_FIX[primary]
        if primary in DROP_PRIMARY:
            applied[f"dropped: {primary}"] += 1
            continue

        m = STROKE_RE.match(primary)
        if m:
            side, high, tech = m.groups()
            lean = parts[1] if len(parts) > 1 else None
            if lean in LEAN_FIX:
                applied[f"lean: {lean} -> {LEAN_FIX[lean]}"] += 1
                lean = LEAN_FIX[lean]
            feet = parts[2] if len(parts) > 2 else None

            n = NFRAMES.get(vid, 10**9)
            w0, w1 = frame - PRE_FRAMES, frame + POST_FRAMES
            strokes.append({
                "stroke_id":  f"{vid}_{frame:07d}",
                "video_id":   vid,
                "fold":       VIDEO2FOLD[vid],
                "orig_split": "train" if vid.startswith("game") else "test",
                "frame_120":  frame,
                "side":       side,
                "high_level": high,
                "technique":  tech,
                "shot_class": TECH2CLASS[tech],
                "lean":       lean,
                "feet":       feet,
                "win_start":  w0,
                "win_end":    w1,
                "win_ok":     bool(w0 >= 0 and w1 < n),
                "pad_start":  max(0, -w0),
                "pad_end":    max(0, w1 - (n - 1)),
                "raw":        str(raw).strip(),
            })
            continue

        if primary in PLAIN_EVENTS:
            events.append({"video_id": vid, "fold": VIDEO2FOLD[vid],
                           "frame_120": frame, "event": primary})
            continue

        r = RALLY_RE.match(primary)
        if r:
            rallies.append({"video_id": vid, "fold": VIDEO2FOLD[vid],
                            "frame_120": frame,
                            "player": r.group(1), "outcome": r.group(2)})
            continue

        unknown[primary] += 1

sdf = pd.DataFrame(strokes).sort_values(["video_id", "frame_120"])
edf = pd.DataFrame(events).sort_values(["video_id", "frame_120"])
rdf = pd.DataFrame(rallies).sort_values(["video_id", "frame_120"])

## 7 · Report & validate

Hard assertions — this cell **fails loudly** rather than letting bad data through to Stage 1.

In [7]:
print("=" * 78)
print("CORRECTIONS APPLIED")
print("=" * 78)
for k, v in applied.items():
    print(f"  {v:>3}x  {k}")
if unknown:
    print(f"\n  !! Unrecognised tokens (ignored): {dict(unknown)}")

print("\n" + "=" * 78)
print("RECONCILIATION")
print("=" * 78)
got = sdf["technique"].value_counts().to_dict()
ok = True
for t in sorted(EXPECTED_TECH, key=lambda x: -EXPECTED_TECH[x]):
    g, e = got.get(t, 0), EXPECTED_TECH[t]
    ok &= g == e
    print(f"  {t:<8} {g:>5} / {e:<5} {'' if g == e else '  <-- MISMATCH'}")
print(f"  {'TOTAL':<8} {len(sdf):>5} / {EXPECTED_STROKES}")
assert len(sdf) == EXPECTED_STROKES and ok, \
    "Reconciliation failed — do not proceed to Stage 1."
print("\n  Reconciles exactly with the paper.")

print("\n" + "=" * 78)
print("FOLD LAYOUT")
print("=" * 78)
piv = pd.crosstab(sdf["fold"], sdf["shot_class"])
piv = piv.reindex(columns=[c for c in TAX["classes"] if c in piv.columns],
                  fill_value=0)
piv["TOTAL"] = piv.sum(axis=1)
piv["videos"] = [", ".join(FOLDS[f]) for f in piv.index]
print(piv.to_string())

zero = [(f, c) for f in piv.index for c in piv.columns[:-2] if piv.loc[f, c] == 0]
assert not zero, f"Fold(s) with a zero-support class: {zero}"
print("\n  Every fold has all 4 classes.")
thin = {f: int(piv.loc[f].iloc[:4].min()) for f in piv.index
        if piv.loc[f].iloc[:4].min() < 15}
if thin:
    print(f"  Low min-class support (report, don't weight): {thin}")

print("\n" + "=" * 78)
print(f"WINDOW GEOMETRY  (-{PRE_FRAMES}, +{POST_FRAMES} @ {NATIVE_FPS}fps "
      f"= {(PRE_FRAMES+POST_FRAMES)/NATIVE_FPS:.2f}s, "
      f"{PRE_FRAMES+POST_FRAMES+1} frames)")
print("=" * 78)
bad = sdf[~sdf["win_ok"]]
print(f"  Windows fully inside video : {sdf['win_ok'].sum()} / {len(sdf)}")
if len(bad):
    print(f"  Needing edge padding       : {len(bad)}")
    for _, r in bad.iterrows():
        which = "start" if r.pad_start else "end"
        print(f"    {r.stroke_id}  frame {r.frame_120}  pad {which} "
              f"{max(r.pad_start, r.pad_end)} frames")
    print("  -> Stage 2 will edge-replicate these. Harmless at this count.")

print("\n" + "=" * 78)
print("SIDE / HANDEDNESS PREP")
print("=" * 78)
print(pd.crosstab(sdf["video_id"], sdf["side"]).to_string())
print("\n  'side' = which end of the table, NOT handedness.")
print("  Handedness still needs the player registry (Step 4).")

print("\n" + "=" * 78)
print("AUXILIARY LABELS  (Stage 5 multi-task heads)")
print("=" * 78)
for c in ["high_level", "lean", "feet"]:
    n = sdf[c].notna().sum()
    print(f"  {c:<11} {n}/{len(sdf)} ({100*n/len(sdf):.1f}%)  "
          f"{dict(sdf[c].value_counts())}")

print("\n" + "=" * 78)
print("EVENT + RALLY LAYERS")
print("=" * 78)
print(f"  events : {len(edf):>5}  {dict(edf['event'].value_counts())}")
print(f"  rallies: {len(rdf):>5}  {dict(rdf['outcome'].value_counts())}")
contact = len(sdf) + int((edf['event'] == 'empty_event').sum())
print(f"\n  Stage 6 contact positives = {len(sdf)} typed strokes "
      f"+ {int((edf['event']=='empty_event').sum())} untyped empty_event "
      f"= {contact}")


CORRECTIONS APPLIED
    1x  lean: back_heavyn -> back_heavy
    1x  primary: xright_backhand_chop -> right_backhand_chop
    1x  dropped: point

RECONCILIATION
  loop       583 / 583   
  serve      290 / 290   
  push       279 / 279   
  block      187 / 187   
  flick       65 / 65    
  chop        31 / 31    
  smash       12 / 12    
  lob         10 / 10    
  TOTAL     1457 / 1457

  Reconciles exactly with the paper.

FOLD LAYOUT
shot_class  serve  attack  control  defence  TOTAL                                  videos
fold                                                                                      
A              24      72       13       52    161                                  game_1
B              92     195       64       48    399                                  game_2
C              30      65       23       35    153                                  game_3
D              30      53       53       37    173                                  game_4
E          

## 8 · Write outputs

Parquet where available, CSV fallback otherwise — a missing `pyarrow` shouldn't discard the whole run.

In [8]:
def save(df, stem):
    """Parquet if an engine is available, else CSV. Never lose the work."""
    try:
        df.to_parquet(META / f"{stem}.parquet", index=False)
        return f"{stem}.parquet"
    except Exception as e:
        df.to_csv(META / f"{stem}.csv", index=False)
        print(f"  (parquet unavailable: {type(e).__name__} — wrote CSV instead)")
        return f"{stem}.csv"


written = [save(sdf, "strokes"), save(edf, "events"), save(rdf, "rallies")]

(META / "folds.json").write_text(json.dumps({
    "folds": FOLDS,
    "video2fold": VIDEO2FOLD,
    "window": {"native_fps": NATIVE_FPS, "pre": PRE_FRAMES,
               "post": POST_FRAMES, "n_frames": PRE_FRAMES + POST_FRAMES + 1},
    "taxonomy_version": TAX.get("version"),
    "corrections": dict(applied),
}, indent=2))

print("\n" + "=" * 78)
for name, df in zip(written, [sdf, edf, rdf]):
    print(f"  {name:<20} {len(df)} rows")
print(f"  {'folds.json':<20}")
print("=" * 78)
print("\n  STAGE 0 COMPLETE.")



  strokes.parquet      1457 rows
  events.parquet       3206 rows
  rallies.parquet      281 rows
  folds.json          

  STAGE 0 COMPLETE.


## 9 · Sanity read-back

So future-you can open this notebook and confirm it worked without re-running the ingest.

In [9]:
def load(stem):
    p = META / f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META / f"{stem}.csv")

s = load("strokes")
print(f"strokes: {len(s)} rows, {s.stroke_id.nunique()} unique ids")
print(f"columns: {list(s.columns)}\n")

print("class x fold")
print(pd.crosstab(s.fold, s.shot_class).to_string())

print("\nclass totals")
print(s.shot_class.value_counts().to_string())

print(f"\nwindows needing edge padding: {(~s.win_ok).sum()}")
print(f"\nfolds.json window: {json.loads((META/'folds.json').read_text())['window']}")

s.head(3)

strokes: 1457 rows, 1457 unique ids
columns: ['stroke_id', 'video_id', 'fold', 'orig_split', 'frame_120', 'side', 'high_level', 'technique', 'shot_class', 'lean', 'feet', 'win_start', 'win_end', 'win_ok', 'pad_start', 'pad_end', 'raw']

class x fold
shot_class  attack  control  defence  serve
fold                                       
A               72       13       52     24
B              195       64       48     92
C               65       23       35     30
D               53       53       37     30
E               96       94        6     52
F               76       14       33     32
G              103       18       17     30

class totals
shot_class
attack     660
serve      290
control    279
defence    228

windows needing edge padding: 2

folds.json window: {'native_fps': 120, 'pre': 60, 'post': 36, 'n_frames': 97}


,stroke_id,video_id,fold,orig_split,frame_120,side,high_level,technique,shot_class,lean,feet,win_start,win_end,win_ok,pad_start,pad_end,raw
0,game_1_0000058,game_1,A,train,58,left,forehand,lob,defence,right_leaning,both_feet_planted,-2,94,False,2,0,left_forehand_lob right_leaning both_feet_planted
1,game_1_0002185,game_1,A,train,2185,right,forehand,serve,serve,neutral,both_feet_planted,2125,2221,True,0,0,right_forehand_serve neutral both_feet_planted
2,game_1_0002245,game_1,A,train,2245,left,backhand,loop,attack,neutral,both_feet_planted,2185,2281,True,0,0,left_backhand_loop neutral both_feet_planted
